# 24 · Tool authentication and credentials

Model authentication (ADC) and authentication to a business API are separate. Use ADK auth configuration for the latter, never an API key embedded in the prompt.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Lite model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from fastapi.openapi.models import HTTPBearer
from google.adk.auth.auth_credential import AuthCredential, AuthCredentialTypes, HttpAuth, HttpCredentials
from google.adk.tools.openapi_tool import OpenAPIToolset
from local_api import start_order_api, order_spec

# A fake credential for a local teaching server, not a real secret.
demo_token = "classroom-demo-token"
scheme = HTTPBearer()
credential = AuthCredential(auth_type=AuthCredentialTypes.HTTP,
    http=HttpAuth(scheme="bearer", credentials=HttpCredentials(token=demo_token)))
server, thread, base_url = start_order_api(bearer_token=demo_token)
toolset = OpenAPIToolset(spec_dict=order_spec(base_url, secured=True), auth_scheme=scheme, auth_credential=credential)
try:
    lab = await make_lab(llm("authenticated_api", "Use the API tool for order facts.", tools=[toolset]))
    await ask(lab, "What is the status of O1002?")
finally:
    await toolset.close()
    server.shutdown()
    server.server_close()
    thread.join(timeout=5)


For OAuth, the application handles ADK's credential request, redirects the user to the provider, and returns the resulting credential through the matching function response. That flow requires a real provider registration and redirect URI; this local bearer example exercises credential injection without those accounts.


## Try it

Call the local API directly without a bearer token and observe HTTP 401.


## Reference

[Official ADK documentation](https://adk.dev/tools-custom/authentication/). Shared configuration: `config.json`. No environment activation or login cells are needed.
